In [6]:
import os
import pandas as pd

HF_HOME = os.environ.get("HF_HOME", os.path.expanduser("~"))

train_path = os.path.join(HF_HOME, "data", "maze_byte_dance", "train.parquet")
test_path = os.path.join(HF_HOME, "data", "maze_byte_dance", "test.parquet")

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train columns:", train_df.columns)
print("Test columns:", test_df.columns)


Train shape: (40000, 5)
Test shape: (775, 5)
Train columns: Index(['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'], dtype='object')
Test columns: Index(['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'], dtype='object')


In [7]:
# Print a sample prompt and reward model from the train set

# Show the first row as a sample
sample = train_df.iloc[0]

# Print the prompt (assuming column names, adjust if needed)
print("Sample prompt:")
if "prompt" in sample:
    print(sample["prompt"])
elif "input" in sample:
    print(sample["input"])
else:
    print("Prompt column not found. Available columns:", train_df.columns)

# Print the reward model (assuming column names, adjust if needed)
print("\nSample reward model output:")
if "reward" in sample:
    print(sample["reward"])
elif "reward_model" in sample:
    print(sample["reward_model"])
else:
    print("Reward column not found. Available columns:", train_df.columns)


Sample prompt:
[{'content': "You need to solve the follow maze.\n'*' denotes the wall that you cannot walk through, '.' denotes available area that you can walk through.\n'S' denotes the starting point, 'E' denotes the destination.\nYou need to start from the starting point and cross through the available area to reach the destination.\nThere are four movement actions, including Left, Right, Up, Down.\nYou need to use L to denote Left movement, R to denote Right movement, U to denote Up movement, and D to denote Down movement.\nYou can analysis the maze to find the correct path, and you should write the final path in the <answer> </answer>, e.g., <answer> LLRRDUL </answer>.\n\n## Maze\n*********\n*.*S....*\n*.*.***.*\n*.*.*...*\n*.*.*.***\n*...*E..*\n*******.*\n*.......*\n*********\n\nNow Try to analyze the maze and put the final path in the <answer> </answer>.", 'role': 'user'}]

Sample reward model output:
{'ground_truth': '*********\n*.*S....*\n*.*.***.*\n*.*.*...*\n*.*.*.***\n*...*

In [8]:
## Sample example from the training run

In [9]:


import sys
sys.path.append("/users/stalaei/code/pass_at_k")
from custom.maze_verifier import extract_answer_maze, compute_score

sample = {"input": "system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.\nuser\nYou need to solve the follow maze.\n'*' denotes the wall that you cannot walk through, '.' denotes available area that you can walk through.\n'S' denotes the starting point, 'E' denotes the destination.\nYou need to start from the starting point and cross through the available area to reach the destination.\nThere are four movement actions, including Left, Right, Up, Down.\nYou need to use L to denote Left movement, R to denote Right movement, U to denote Up movement, and D to denote Down movement.\nYou can analysis the maze to find the correct path, and you should write the final path in the <answer> </answer>, e.g., <answer> LLRRDUL </answer>.\n\n## Maze\n*******\n*.....*\n*.*.***\n*S*.*.*\n*.*.*.*\n*E*...*\n*******\n\nNow Try to analyze the maze and put the final path in the <answer> </answer>.\nassistant\n", "output": "To solve this maze, let's start from the 'S' and move step by step towards the 'E', keeping track of the movements.\n\n1. Start at 'S' (2, 2).\n2. Move Right (R) to (2, 3).\n3. Move Right (R) to (2, 4).\n4. Move Down (D) to (3, 4).\n5. Move Down (D) to (4, 4).\n6. Move Right (R) to (4, 5).\n7. Move Right (R) to (4, 6).\n8. Move Up (U) to (3, 6).\n9. Move Up (U) to (2, 6).\n10. Move Right (R) to (2, 7).\n11. Move Right (R) to (2, 8).\n12. Move Down (D) to (3, 8).\n13. Move Down (D) to (4, 8).\n14. Move Down (D) to (5, 8).\n\nThe final path is: <answer>RRDDRRUUURRRDDD</answer>", "gts": "*******\n*.....*\n*.*.***\n*S*.*.*\n*.*.*.*\n*E*...*\n*******", "score": 0.0, "step": 1, "reward": 0.0}
print(sample["input"])
print("-"*100)

print(f"Model's output:\n{sample['output']}")
print("-"*100)
output = extract_answer_maze(sample["output"])
print(f"Extracted output:\n{output}")


score = compute_score(output, sample["gts"])
print("Score for output:", score)

print("Reward:", sample["reward"], "\n")
print("-"*100)
correct_output = "<answer>DD</answer>"
correct_output = extract_answer_maze(correct_output)
print(f"Correct output:\n{correct_output}")

score = compute_score(correct_output, sample["gts"])
print("Score for correct output:", score)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
You need to solve the follow maze.
'*' denotes the wall that you cannot walk through, '.' denotes available area that you can walk through.
'S' denotes the starting point, 'E' denotes the destination.
You need to start from the starting point and cross through the available area to reach the destination.
There are four movement actions, including Left, Right, Up, Down.
You need to use L to denote Left movement, R to denote Right movement, U to denote Up movement, and D to denote Down movement.
You can analysis the maze to find the correct path, and you should write the final path in the <answer> </answer>, e.g., <answer> LLRRDUL </answer>.

## Maze
*******
*.....*
*.*.***
*S*.*.*
*.*.*.*
*E*...*
*******

Now Try to analyze the maze and put the final path in the <answer> </answer>.
assistant

----------------------------------------------------------------------------------------------------
Model's output:

##

## Testing a sample from the dataset in inference

In [10]:
import sys
sys.path.append("/users/stalaei/code/pass_at_k")
from custom.model_inference import load_model, generate_text
import os
HF_HOME = os.environ.get("HF_HOME", os.path.expanduser("~"))

model = load_model(f"{HF_HOME}/models/Qwen2_5-7B-Instruct")
output = generate_text(model, "What is the capital of France?")
print(output)

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.61s/it]


The capital of France is Paris.


In [11]:
model_output = generate_text(model, sample["input"], temperature=1.0, top_p=0.95)
print(model_output)

To solve this maze, we will start from the 'S' (starting point) and move step by step towards the 'E' (destination) using valid movements (Left, Right, Up, Down).

Here's the step-by-step analysis of the maze:

1. Start at 'S' (2, 3).
2. Move Right to (2, 4).
3. Move Right to (2, 5).
4. Move Down to (3, 5).
5. Move Down to (4, 5).
6. Move Down to (5, 5).
7. Move Right to (5, 6).
8. Move Down to (5, 7).

The path can be represented as: RRRDDD

Therefore, the final path is:

<answer>RRRDD</answer>


In [12]:
another_sample = {"input": "system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.\nuser\nYou need to solve the follow maze.\n'*' denotes the wall that you cannot walk through, '.' denotes available area that you can walk through.\n'S' denotes the starting point, 'E' denotes the destination.\nYou need to start from the starting point and cross through the available area to reach the destination.\nThere are four movement actions, including Left, Right, Up, Down.\nYou need to use L to denote Left movement, R to denote Right movement, U to denote Up movement, and D to denote Down movement.\nYou can analysis the maze to find the correct path, and you should write the final path in the <answer> </answer>, e.g., <answer> LLRRDUL </answer>.\n\n## Maze\n*******\n*.....*\n*****.*\n*...*S*\n*.*.*.*\n*.*..E*\n*******\n\nNow Try to analyze the maze and put the final path in the <answer> </answer>.\nassistant\n", "output": "To solve this maze, let's start from the 'S' and try to find a path to the 'E' by analyzing the available paths step by step.\n\n1. Start at 'S' (3,2).\n2. Move Down to (4,2) and then Right to (4,3).\n3. Move Down to (5,3) and then Right to (5,4).\n4. Move Down to (5,5) and then Right to (5,6).\n5. Move Down to (5,7) and then Left to (5,6).\n6. Move Up to (4,6) and then Left to (4,5).\n7. Move Up to (3,5) and then Left to (3,4).\n8. Move Up to (2,4) and then Left to (2,3).\n9. Move Up to (1,3) and then Right to (1,4).\n10. Move Right to (1,5) and then Right to (1,6).\n11. Move Right to (1,7) and then Up to (2,7).\n12. Move Up to (3,7) and then Up to (4,7).\n13. Move Up to (5,7) and then Left to (5,6).\n14. Move Left to (5,5) and then Left to (5,4).\n15. Move Left to (5,3) and then Up to (4,3).\n16. Move Up to (3,3) and then Up to (2,3).\n17. Move Right to (2,4) and then Right to (2,5).\n18. Move Right to (2,6) and then Right to (2,7).\n19. Move Up to (1,7) and then Up to (0,7).\n\nThe path can be represented as follows:\n<answer>DDRRRUUULLLRRRRUUUDDDRRRUUU</answer>\n\nThis path leads from the starting point 'S' to the destination 'E'.", "gts": "*******\n*.....*\n*****.*\n*...*S*\n*.*.*.*\n*.*..E*\n*******", "score": 0.0, "step": 1, "reward": 0.0}
print(another_sample["input"])
print("-"*100)
model_output = generate_text(model, another_sample["input"], temperature=1.0, top_p=0.95)
print(f"Model's output:\n{model_output}")
print("-"*100)
output = extract_answer_maze(model_output)
print(f"Extracted output:\n{output}")
print("-"*100)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
You need to solve the follow maze.
'*' denotes the wall that you cannot walk through, '.' denotes available area that you can walk through.
'S' denotes the starting point, 'E' denotes the destination.
You need to start from the starting point and cross through the available area to reach the destination.
There are four movement actions, including Left, Right, Up, Down.
You need to use L to denote Left movement, R to denote Right movement, U to denote Up movement, and D to denote Down movement.
You can analysis the maze to find the correct path, and you should write the final path in the <answer> </answer>, e.g., <answer> LLRRDUL </answer>.

## Maze
*******
*.....*
*****.*
*...*S*
*.*.*.*
*.*..E*
*******

Now Try to analyze the maze and put the final path in the <answer> </answer>.
assistant

----------------------------------------------------------------------------------------------------
Model's output:

## Running eval on the test train set

In [13]:
import pandas as pd

df = test_df


In [16]:
from custom.maze_verifier import extract_answer_maze, compute_score

results = []
batch_size = 100
num_samples = 100
temperature = 1.0
top_p = 0.95
max_length = 8096

def extract_prompt(row):
    prompt_data = row['prompt']
    if isinstance(prompt_data, list) and len(prompt_data) > 0:
        user_msg = None
        for msg in prompt_data:
            if isinstance(msg, dict) and msg.get("role", "").lower() == "user":
                user_msg = msg.get("content", None)
                break
        if user_msg is None:
            user_msg = prompt_data[0].get("content", None)
        prompt = user_msg
    elif isinstance(prompt_data, dict):
        prompt = prompt_data.get("content", None)
    else:
        prompt = str(prompt_data)
    return prompt

def extract_maze(row):
    reward_model = row.get("reward_model", {})
    if isinstance(reward_model, dict):
        maze = reward_model.get("ground_truth", None)
    else:
        maze = None
    return maze

def extract_gt_answer(row, reward_model):
    gt_answer = None
    if "output" in row:
        gt_answer = extract_answer_maze(row["output"])
    elif isinstance(reward_model, dict) and "ground_truth" in reward_model:
        gt_answer = extract_answer_maze(reward_model["ground_truth"])
    return gt_answer

rows = df.head(num_samples)
for batch_start in range(0, len(rows), batch_size):
    batch_rows = rows.iloc[batch_start:batch_start+batch_size]
    prompts = []
    mazes = []
    idxs = []
    gt_answers = []
    valid_rows = []
    for idx, row in batch_rows.iterrows():
        prompt = extract_prompt(row)
        maze = extract_maze(row)
        if maze is None or prompt is None:
            print(f"Sample {idx} missing maze or prompt text, skipping.")
            continue
        prompts.append(prompt)
        mazes.append(maze)
        idxs.append(idx)
        reward_model = row.get("reward_model", {})
        gt_answer = extract_gt_answer(row, reward_model)
        gt_answers.append(gt_answer)
        valid_rows.append(row)
        # print(f"Prompt:\n{prompt}")
        # print(f"Reward model:\n{reward_model}")

    if not prompts:
        continue

    # Batch call to generate_text
    model_outputs = generate_text(model, prompts, 
                                  temperature=temperature, 
                                  top_p=top_p, 
                                  max_length=max_length)

    # If generate_text does not support batch, fallback to list comprehension
    if not isinstance(model_outputs, list):
        model_outputs = [model_outputs]

    for i, (idx, prompt, maze, model_output, gt_answer) in enumerate(zip(idxs, prompts, mazes, model_outputs, gt_answers)):
        extracted = extract_answer_maze(model_output)
        score = compute_score(model_output, maze)
        results.append({
            "idx": idx,
            "prompt": prompt,
            "maze": maze,
            "model_output": model_output,
            "extracted_answer": extracted,
            "score": score,
            "gt_answer": gt_answer,
        })
        print(f"Sample {idx}: Score={score}")
        print(f"Model output:\n{model_output}")
        print(f"Extracted answer: {extracted}")
        print(f"Ground truth answer: {gt_answer}")
        print("-"*80)

# Optionally, show summary statistics
scores = [r["score"] for r in results]
print(f"Average score over {len(scores)} samples: {sum(scores)/len(scores) if scores else 0:.2f}")


KeyboardInterrupt: 

In [ ]:
for res in results:
    
    print(f"Model output: {res['model_output']}")
    print("-"*100)
    print(f"Extracted answer: {res['extracted_answer']}")
    print("-"*100)
    print(f"GTS: {res['gt_answer']}")
    print(f"Score: {res['score']}")
    print("==============="*100)

Model output: 
assistant
To solve this maze, we will start from the 'S' (starting point) and move step-by-step towards the 'E' (destination) by analyzing the maze layout.

Here is the given maze:
```
*********
*.*S....*
*.*.***.*
*.*.*...*
*.*.*.***
*...*E..*
*******.*
*.......*
*********
```

1. Start at S: *.*S....*
2. Move right (R) to avoid walls and blocked paths:
   - Move right to *.*.*...*
3. Continue moving right (R) until blocked by a wall or another obstacle:
   - Move right to *.*.*.***.
4. Move down (D) to navigate around the wall:
   - Move down to *.*.*...*E.*
5. Move left (L) to continue moving around the wall:
   - Move left to *.*.*.*...E
6. Move left (L) to clear the way to the destination:
   - Move left to *.*.*..E...
7. Move left (L) to finally reach the destination:
   - Move left to *.*.*E.....
8. Finally, move down (D) to the open space:
   - Move down to *...*E....
9. Move down (D) one more time to reach the destination:
   - Move down to **..**E....

The path